# Grid run with local backend — status tracking showcase

This notebook demonstrates:
1. Setting up a grid of MESA models with varying parameters.
2. Running them in parallel with the `local` backend.
3. Tracking status with `SQLiteRunRepository`, `LiveProcessRegistry`, and `RepositoryObserver`.
4. All five status APIs: `get_model_status`, `list_active_status`, `list_models_status_snapshot`, `stream_models_status`, and `live_status`.

## 1. Environment and imports

In [12]:
import os
from pathlib import Path
import morpy as mr
from morpy import (
    SQLiteRunRepository,
    LiveProcessRegistry,
    RepositoryObserver,
    GridRunItem,
    run_grid,
    get_model_status,
    list_active_status,
    list_models_status_snapshot,
    stream_models_status,
    live_status,
)
import pandas as pd
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

In [14]:
# Adjust these paths to match your local MESA installation.
MESA_DIR     = Path('/Users/janhenneco/software/mesa/mesa-r24.08.1')
MESASDK_ROOT = Path('/Applications/mesasdk-25.12.1/')
OMP_THREADS  = 1
MESA_VERSION = 'r24.08.1'

# Source workdir to copy models from.
SOURCE_WORKDIR = Path('/Users/janhenneco/GitHub/MOrPy/tests/data_for_tests/r24.08.1/workdirs/single_work')

# Project dir where model directories will be created.
PROJECT_DIR = Path('/Users/janhenneco/GitHub/MOrPy/tests/manual_tests/output_tests/project_single_work')

# SQLite database for run history.
DB_PATH = PROJECT_DIR / 'runs.db'

In [15]:
mconf = mr.setup_mesa_env(mesa_dir=MESA_DIR, mesasdk_root=MESASDK_ROOT, omp_num_threads=OMP_THREADS, version_name=MESA_VERSION)
def_opts = mr.build_defaults_options_from_config(mconf)
print('MESA version:', mconf.mesa_version)

MESA version: MesaVersion(version_name='r24.08.1', scheme='release', revision=None, year=24, month=8, patch=1)


In [16]:
search_results = def_opts.controls.search('overshoot')
mr.preview(search_results, max_desc = 1000000, prefer_pandas=True)


,namelist,name,default,current,description
0,controls,burn_z_mix_region_logT,8.7d0,,max logT in convective region determines burn type for overshooting
1,controls,limit_overshoot_Hp_using_size_of_convection_zone,.true.,,"if false, allow large distance of overshoot for small convective zones."
2,controls,min_overshoot_q,0d0,,"Overshooting is only allowed at locations with mass ``m >= min_overshoot_q * mstar``. E.g., if ``min_overshoot_q = 0.1``, then only the outer 90% by mass can have overshooting. This provides a simple way of suppressing bogus center overshooting in which a small convective region at the core can produce excessively large overshooting because of a large pressure scale height at the center."
3,controls,MLT_option,'TDC',,"Options are: + 'none' : just give radiative values with no mixing. + 'TDC' : Time-dependent convection based on the Khufuss 1986 model. Reduces to Cox at long times. + 'Cox' : MLT as developed in Cox & Giuli 1968, Chapter 14. + 'ML1' : Bohm-Vitense 1958 + 'ML2' : Bohm and Cassinelli 1971 + 'Mihalas' : Mihalas 1978, Kurucz 1979 + 'Henyey' : Henyey, Vardya, and Bodenheimer 1965 The 'Cox' and 'TDC' options assumes optically thick material. The other options are various ways of extending to include optically thin material. Note that TDC does not respect the ``alt_scale_height`` option, and continues to use ``h = P / rho g`` even if that flag is set. We caution that combining different mixing models in a stellar evolution calculation might lead to physically inconsistent solutions, because the different models have been developed separately, and their underlying assumptions might not be compatible with each other. Examples include combining the newly implemented time-dependent local limit convection model ``TDC`` with an overshooting model, or combining ``TDC`` with other models for chemical composition gradients (predictive mixing or convective premixing), rotation, etc. (MESA VI)"
4,controls,overshoot_alpha,-1,,"NOTE: In addition to giving these 'f' parameters non-zero values, you should also check the settings for ``mass_for_overshoot_full_on`` and ``mass_for_overshoot_full_off``. The value of Hp for overshooting is limited to the radial thickness of the convection zone divided by ``overshoot_alpha``. only used when > 0. if <= 0, then use ``mixing_length_alpha`` instead."
5,controls,overshoot_bdy_loc(:),'',,"``bottom``, ``top``, ``any``"
6,controls,overshoot_brunt_B_max,0d0,,Terminate overshoot region when encounter stabilizing composition gradient where (unsmoothed) ``brunt_B`` is greater than this limit. (<= 0 means ignore this limit) note: both ``brunt_B`` and ``gradL_composition_term`` come from ``unsmoothed_brunt_B`` and differ only in optional smoothing. (see ``num_cells_for_smooth_brunt_B`` and ``num_cells_for_smooth_gradL_composition_term``).
7,controls,overshoot_D0(:),0d0,,"The switch from convective mixing to overshooting happens at a distance f0*Hp into the convection zone from the estimated location where ``grad_ad = grad_rad``, where Hp is the pressure scale height at that location. A value <= 0 for f0 is a mistake -- you are required to set f0 as well as f. take a look at the following from an email concerning this: Overshooting works by taking the diffusion mixing coefficient at the edge of the convection zone and extending it beyond the zone. But -- and here's the issue -- at the exact edge of the zone the mixing coefficient goes to 0. So we don't want that. Instead we want the value of the mixing coeff NEAR the edge, but not AT the edge. The ""f0"" parameter determines the exact meaning of ""near"" for this. It tells the code how far back into the zone to go in terms of scale height. The overshooting actually begins at the location determined by f0 back into the convection zone rather than at the edge where the diffusion coeff is ill-defined. So, for example, if you want overshooting of 0.2 scale heights beyond the normal edge, you might want to back up 0.05 scale heights to ge

## 2. Build the model grid

Import a base model from disk, then create copies with different `initial_mass` values.
Each copy becomes one `GridRunItem`.

In [22]:
# Import base model from the source workdir.
base_model = mr.Model.empty(name='base')
mr.import_inlists_from_dir(model=base_model, source_dir=SOURCE_WORKDIR,
                            default_options=def_opts, config=mconf)

# Parameter grid: vary initial stellar mass.
initial_masses = [8.0, 15.0, 20.0, 25.0]

# Add overshooting to inlist
base_model.inlists['inlist_project'].set_options(defaults=def_opts, controls={'overshoot_f(1)': 0.055,
                                                                              'overshoot_f0(1)': 0.005,
                                                                              'overshoot_bdy_loc(1)': 'top',
                                                                              'overshoot_scheme(1)': 'exponential',
                                                                              'overshoot_zone_loc(1)': 'core',
                                                                              'overshoot_zone_type(1)': 'burn_H'})

# Create project directory and compile once.
project_dir = mr.create_projectdir(projectdir_path=PROJECT_DIR, mode='star')
compile_result = mr.compile_project(projectdir_path=project_dir, mode='star')
print(compile_result.pretty(2000))

success: True
return_code: 0
duration_seconds: 1.4619423749973066
mode: star
executable_path: /Users/janhenneco/GitHub/MOrPy/tests/manual_tests/output_tests/project_single_work/star
model_scripts_updated: ['/Users/janhenneco/GitHub/MOrPy/tests/manual_tests/output_tests/project_single_work/mass_15.0/rn', '/Users/janhenneco/GitHub/MOrPy/tests/manual_tests/output_tests/project_single_work/mass_15.0/re', '/Users/janhenneco/GitHub/MOrPy/tests/manual_tests/output_tests/project_single_work/mass_20.0/rn', '/Users/janhenneco/GitHub/MOrPy/tests/manual_tests/output_tests/project_single_work/mass_20.0/re', '/Users/janhenneco/GitHub/MOrPy/tests/manual_tests/output_tests/project_single_work/mass_25.0/rn', '/Users/janhenneco/GitHub/MOrPy/tests/manual_tests/output_tests/project_single_work/mass_25.0/re', '/Users/janhenneco/GitHub/MOrPy/tests/manual_tests/output_tests/project_single_work/mass_8.0/rn', '/Users/janhenneco/GitHub/MOrPy/tests/manual_tests/output_tests/project_single_work/mass_8.0/re']
clea

In [18]:
mr.preview(base_model.inlists['inlist_project'].options)

,namelist,name,default,current,description
0,star_job,create_pre_main_sequence_model,.false.,.true.,"If true, the code will create a starting model with uniform composition, a core temperature below..."
1,star_job,save_model_when_terminate,.false.,.false.,save final model when a run terminates only happens if satisfy required termination code if strin...
2,star_job,save_model_filename,'undefined','15M_at_TAMS.mod',saved model root filename
3,star_job,pgstar_flag,.false.,.false.,"if true, activates pgplot output"
4,kap,use_Type2_opacities,.true.,.true.,Select whether to use Type2 opacity tables (see :ref:`kap/defaults:kap_CO_prefix`). Even when tru...
5,kap,Zbase,-1,0.02,The MESA/kap parameters are given default values here. The actual values as modified by your inli...
6,controls,initial_mass,1,15,"NOTE: if you are loading a saved model, then the following initial values are NOT USED to modify ..."
7,controls,initial_z,0.02d0,0.02,initial metallicity for create pre-ms and create initial model ``initial_z`` can be any value fro...
8,controls,Lnuc_div_L_zams_limit,0.9d0,0.99d0,"defines ""near zams"" -- note: must also set ``stop_near_zams``"
9,controls,stop_near_zams,.false.,.false.,"if true, stop if Lnuc/L > ``Lnuc_div_L_zams_limit``"


In [23]:
items = []
for mass in initial_masses:
    model = base_model.copy(model_copy_name=f'mass_{mass:.1f}')

    # Adjust the inlist option that controls initial mass.
    # Replace 'inlist' with the actual inlist file name in your workdir.
    model.inlists['inlist_project'].set_option(
        name='initial_mass', value=mass, defaults=def_opts, namelist='controls'
    )

    # Export the model directory (one per mass).
    mr.create_modeldir(model=model, projectdir_path=project_dir, overwrite=True)

    items.append(GridRunItem(model=model))

print(f'{len(items)} models prepared.')
for item in items:
    print(f'  {item.model.name}  →  {item.model.model_path}')

4 models prepared.
  mass_8.0  →  /Users/janhenneco/GitHub/MOrPy/tests/manual_tests/output_tests/project_single_work/mass_8.0
  mass_15.0  →  /Users/janhenneco/GitHub/MOrPy/tests/manual_tests/output_tests/project_single_work/mass_15.0
  mass_20.0  →  /Users/janhenneco/GitHub/MOrPy/tests/manual_tests/output_tests/project_single_work/mass_20.0
  mass_25.0  →  /Users/janhenneco/GitHub/MOrPy/tests/manual_tests/output_tests/project_single_work/mass_25.0


## 3. Set up persistence and the observer

- `SQLiteRunRepository` stores durable run history in `runs.db`.
- `LiveProcessRegistry` tracks which subprocesses are currently alive.
- `RepositoryObserver` bridges run events to the repository automatically.

In [24]:
if DB_PATH.exists():
    DB_PATH.unlink()

repo          = SQLiteRunRepository(db_path=DB_PATH)
live_registry = LiveProcessRegistry()
observer      = RepositoryObserver(repo=repo)

# Pre-register models so they appear in status queries immediately.
for item in items:
    repo.upsert_model(item.model)

print(f'Repository initialised at: {DB_PATH}')

Repository initialised at: /Users/janhenneco/GitHub/MOrPy/tests/manual_tests/output_tests/project_single_work/runs.db


## 4. Launch the grid run

`run_grid` with `backend='local'` runs up to `max_workers` models in parallel
using a FIFO queue. It blocks until all models have finished.

In [25]:
# grid_result = run_grid(
#     items,
#     backend='local',
#     max_workers=6,           # run at most 2 models simultaneously
#     sync_before_run=True,
#     default_options=def_opts,
#     config=mconf,
#     capture_output=True,
#     observer=observer,
#     live_registry=live_registry,
# )

# print('Grid run complete.')
# print(f'  Total:     {grid_result.total}')
# print(f'  Completed: {grid_result.completed}')
# print(f'  Failed:    {grid_result.failed}')
# print(f'  Cancelled: {grid_result.cancelled}')
# print(f'  Duration:  {grid_result.duration_seconds:.1f}s')

# non-blocking start
handle = mr.launch_grid(
    items,
    backend="local",
    max_workers=6,
    sync_before_run=True,
    default_options=def_opts,
    config=mconf,
    capture_output=True,
    observer=observer,
    live_registry=live_registry,
)

In [27]:
# while grid is running, monitor status
# import itertools
# for rows in itertools.islice(
#     mr.stream_models_status(repo, live_registry=live_registry, interval_seconds=3.0),
#     10,
# ):
#     print([(r.model_name, r.state, r.pid, r.return_code) for r in rows])

# collect final result
grid_result = mr.wait_for_grid(handle)

## 5. Status tracking

### 5a. `get_model_status` — single-model targeted query

Use this when you want to inspect one specific model in detail.

In [ ]:
target_model = items[0].model
status = get_model_status(target_model.model_id, repo, live_registry=live_registry)

print(f'Model:      {status.model_name}')
print(f'State:      {status.state}')
print(f'Return code:{status.return_code}')
print(f'Duration:   {status.duration_seconds:.1f}s')
print(f'Started at: {status.started_at}')
print(f'Finished:   {status.finished_at}')
if status.last_error:
    print(f'Last error: {status.last_error}')

### 5b. `list_active_status` — models currently in progress

After the grid completes this list will be empty. Run this during
a grid to see only the models that are still queued or running.

In [11]:
active = list_active_status(repo, live_registry=live_registry)

if active:
    for s in active:
        print(f'{s.model_name:20s}  state={s.state}  pid={s.pid}')
else:
    print('No active models (grid has finished).')

mass_25.0             state=running  pid=64869
mass_20.0             state=running  pid=64868
mass_8.0              state=running  pid=64866
mass_15.0             state=running  pid=64867


### 5c. `list_models_status_snapshot` — full table of all known models

Returns one row per model known to the repository, sorted by name.
Useful for dashboards and post-run summaries.

In [12]:
snapshot = list_models_status_snapshot(repo, live_registry=live_registry)

print(f'{"Model":<20} {"State":<12} {"RC":>4} {"Duration (s)":>14} {"Last error"}')
print('-' * 70)
for s in snapshot:
    dur  = f'{s.duration_seconds:.1f}' if s.duration_seconds is not None else '-'
    err  = (s.last_error or '')[:40]
    rc   = str(s.return_code) if s.return_code is not None else '-'
    print(f'{s.model_name:<20} {s.state:<12} {rc:>4} {dur:>14}  {err}')

Model                State          RC   Duration (s) Last error
----------------------------------------------------------------------
mass_15.0            running         1            0.0  
mass_20.0            running         1            0.0  
mass_25.0            running         1            0.0  
mass_8.0             running         1            0.0  


Optionally use `pandas` for a nicer table view:

In [13]:
try:
    import pandas as pd
    pd.set_option('display.max_colwidth', None)

    df = pd.DataFrame([
        {
            'model':        s.model_name,
            'state':        s.state,
            'return_code':  s.return_code,
            'duration_s':   round(s.duration_seconds, 1) if s.duration_seconds else None,
            'started_at':   s.started_at,
            'finished_at':  s.finished_at,
            'last_error':   s.last_error,
        }
        for s in snapshot
    ])
    display(df)
except ImportError:
    print('pandas not installed — install morpy[preview] for DataFrame support.')

,model,state,return_code,duration_s,started_at,finished_at,last_error
0,mass_15.0,running,1,None,2026-06-03 14:00:20.238434,2026-06-03 14:00:20.238368,None
1,mass_20.0,running,1,None,2026-06-03 14:00:20.247113,2026-06-03 14:00:20.247062,None
2,mass_25.0,running,1,None,2026-06-03 14:00:20.254054,2026-06-03 14:00:20.254009,None
3,mass_8.0,running,1,None,2026-06-03 14:00:20.226207,2026-06-03 14:00:20.226133,None


### 5d. `stream_models_status` — polling generator for custom live monitors

This cell takes **N** snapshots at fixed intervals and prints them.
It is the building block for any custom live-monitoring loop.

In [14]:
import itertools

N_POLLS = 3           # how many snapshots to take
INTERVAL = 5.0        # seconds between polls

for i, rows in enumerate(itertools.islice(
        stream_models_status(repo, live_registry=live_registry, interval_seconds=INTERVAL),
        N_POLLS)):
    print(f'\n--- Poll {i + 1} ---')
    for s in rows:
        print(f'  {s.model_name:<20} {s.state:<12} rc={s.return_code}')


--- Poll 1 ---
  mass_15.0            running      rc=1
  mass_20.0            running      rc=1
  mass_25.0            running      rc=1
  mass_8.0             running      rc=1

--- Poll 2 ---
  mass_15.0            running      rc=1
  mass_20.0            running      rc=1
  mass_25.0            running      rc=1
  mass_8.0             running      rc=1

--- Poll 3 ---
  mass_15.0            running      rc=1
  mass_20.0            running      rc=1
  mass_25.0            running      rc=1
  mass_8.0             running      rc=1


### 5e. `live_status` — continuous terminal monitor

Prints a live-updating status view until you stop it with **Ctrl+C / Cmd+.**

> **Tip:** run this in a separate terminal or as a background thread
> *while* a grid run is in progress to watch models transition states in real time.

In [ ]:
# Uncomment to start the live monitor (stop with Ctrl+C).
# live_status(repo, live_registry=live_registry, interval_seconds=3.0, clear_screen=False)

## 6. Per-run result details

`GridRunResult.results` contains one `RunResult` per model.
Use this for post-run inspection or custom reporting.

In [15]:
for result in grid_result.results:
    status_label = '✓' if result.success else '✗'
    print(f'{status_label}  {result.model_name:<20}  rc={result.return_code}  '
          f'{result.duration_seconds:.1f}s  {result.terminal_state}')

NameError: name 'grid_result' is not defined

## 7. Historical queries from the repository

Because `RepositoryObserver` wrote every event to SQLite, you can query
run history across sessions — not just the current grid run.

In [ ]:
# Most recent 10 runs across all models.
recent = repo.list_recent_runs(limit=10)
print('Recent runs:')
for r in recent:
    print(f'  {r.model_name:<20} {r.terminal_state:<12} {r.start_time}')

In [ ]:
# Full run history for one model.
target = items[0].model
history = repo.get_model_run_history(target.model_id)
print(f'Run history for {target.name} ({len(history)} run(s)):')
for r in history:
    print(f'  run_id={r.metadata.get("run_id", "?")}  state={r.terminal_state}  '
          f'rc={r.return_code}  {r.duration_seconds:.1f}s')